# 🎛️ Vialactée Parameter Tuner

This notebook runs an automated Grid Search over the internal parameters of the `Listener.py` to find the absolute best mathematical configuration for Beat Detection.

In [7]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import librosa
import torch
import torchaudio
import itertools

# Import your Listener
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import core.Listener as ListenerModule

# --- CONFIGURATION ---
AUDIO_FILE = "C:/Users/Users/Desktop/vialactée/vialactee/mp3_files/Palladium.mp3"
TOLERANCE_WINDOW = 0.150  # Seconds.

print(f"Loading '{AUDIO_FILE}' globally for Ground Truth extraction...")
y, sr = librosa.load(AUDIO_FILE, sr=None) 
print("Extracting actual temporal ground truth...")
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
sub_beat_frames = (beat_frames[:-1] + beat_frames[1:]) // 2
true_beat_times = librosa.frames_to_time(beat_frames, sr=sr)
true_subbeat_times = librosa.frames_to_time(sub_beat_frames, sr=sr)
print(f"Ground Truth established! {len(true_beat_times)} beats discovered.")


Loading 'C:/Users/Users/Desktop/vialactée/vialactee/mp3_files/Palladium.mp3' globally for Ground Truth extraction...
Extracting actual temporal ground truth...
Ground Truth established! 538 beats discovered.


In [8]:
class Simulated_Microphone:
    def __init__(self, audio_file_path, bandValues, infos, chromaValues=None):
        self.bandValues = bandValues
        self.chromaValues = chromaValues
        self.nb_of_fft_band = len(self.bandValues)
        
        self.sample_rate = 44100
        self.buffer_size = 1024 
        self.audio_data = np.zeros(self.buffer_size)
        
        waveform, sr = torchaudio.load(audio_file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)
            
        self.full_audio = waveform.numpy().flatten()
        self.total_samples = len(self.full_audio)
        self.current_pos = 0
        
        fft_size = self.buffer_size // 2 + 1
        self.weight_matrix = np.zeros((self.nb_of_fft_band, fft_size))
        
        def hz_to_mel(f): return 2595 * np.log10(1 + f / 700.0)
        def mel_to_hz(m): return 700 * (10**(m / 2595.0) - 1)
        
        lower_mel = hz_to_mel(20)
        upper_mel = hz_to_mel(20000)
        mel_points = np.linspace(lower_mel, upper_mel, self.nb_of_fft_band + 2)
        hz_points = mel_to_hz(mel_points)
        bin_points = np.floor((self.buffer_size + 1) * hz_points / self.sample_rate).astype(int)
        
        for i in range(self.nb_of_fft_band):
            start = min(bin_points[i], fft_size - 1)
            mid = min(bin_points[i + 1], fft_size - 1)
            end = min(bin_points[i + 2], fft_size - 1)
            if mid > start:
                self.weight_matrix[i, start:mid] = np.linspace(0, 1, mid - start, endpoint=False)
            if end > mid:
                self.weight_matrix[i, mid:end] = np.linspace(1, 0, end - mid, endpoint=False)
            band_sum = np.sum(self.weight_matrix[i, :])
            if band_sum > 0:
                self.weight_matrix[i, :] /= band_sum

    def pop_chunk(self, chunk_size=1024):
        if self.current_pos + chunk_size > self.total_samples:
            return False 
        
        incoming = self.full_audio[self.current_pos : self.current_pos + chunk_size]
        self.current_pos += chunk_size
        self.audio_data = np.roll(self.audio_data, -chunk_size)
        self.audio_data[-chunk_size:] = incoming
        return True

    def calculate_fft(self):
        windowed_data = self.audio_data * np.hanning(self.buffer_size)
        fft_result = np.abs(np.fft.rfft(windowed_data))
        scale = 150.0 / (self.buffer_size / 1024.0)
        mel_bands = np.dot(self.weight_matrix, fft_result) * scale
        for i in range(self.nb_of_fft_band):
            self.bandValues[i] = int(mel_bands[i])


In [9]:
def evaluate_beats(alg_beats, true_beats, sub_beats, tolerance=0.150):
    matched_true = set()
    matched_alg_main = set()
    matched_sub = set()
    matched_alg_sub = set()
    
    for i, a_beat in enumerate(alg_beats):
        closest_dist = float('inf')
        closest_j = -1
        
        for j, t_beat in enumerate(true_beats):
            if j in matched_true: continue
            dist = abs(a_beat - t_beat)
            if dist < closest_dist and dist <= tolerance:
                closest_dist = dist
                closest_j = j
                
        if closest_j != -1:
            matched_true.add(closest_j)
            matched_alg_main.add(i)
        else:
            for s, s_beat in enumerate(sub_beats):
                if s in matched_sub: continue
                dist = abs(a_beat - s_beat)
                if dist <= tolerance:
                    matched_sub.add(s)
                    matched_alg_sub.add(i)
                    break

    main_hits = len(matched_alg_main)
    sub_hits = len(matched_alg_sub)
    total_hits = main_hits + sub_hits
    
    precision = total_hits / len(alg_beats) if len(alg_beats) > 0 else 0
    recall = total_hits / len(true_beats) if (len(true_beats) + len(sub_beats)) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return f1_score


In [10]:
class MockTime:
    def __init__(self):
        self.current_time = 0.0
    def time(self):
        return self.current_time

mock_timer = MockTime()
ListenerModule.time.time = mock_timer.time

CHUNK_SIZE_FOR_60FPS = int(44100 / 60.0)
TIME_PER_FRAME = 1.0 / 60.0

def test_parameters(momentum_mult, base_pull, latency, decay):
    # Using the exact variables we exposed in Listener.py's __init__
    infos = {
        "printAsservmentDetails": False, 
        "useMicrophone": True,
        "momentum_mult": momentum_mult,
        "base_pull": base_pull,
        "latency": latency,
        "decay_base": decay
    }
    
    mock_timer.current_time = 0.0
    listener = ListenerModule.Listener(infos)
    
    mic = Simulated_Microphone(AUDIO_FILE, listener.fft_band_values, infos)
    
    listener.hasBeenSilenceCalibrated = True
    listener.hasBeenBBCalibrated = True
    listener.calibrate_silence = lambda: None
    listener.calibrate_bb = lambda: None
    
    algorithmic_beats = []
    last_logged_beat_count = 0
    
    while mic.pop_chunk(CHUNK_SIZE_FOR_60FPS):
        mic.calculate_fft()
        listener.update()
        
        if getattr(listener, 'beat_count', 0) > last_logged_beat_count:
            algorithmic_beats.append(mock_timer.current_time)
            last_logged_beat_count = listener.beat_count
            
        mock_timer.current_time += TIME_PER_FRAME
        
    f1 = evaluate_beats(np.array(algorithmic_beats), true_beat_times, true_subbeat_times, TOLERANCE_WINDOW)
    return f1


In [12]:
import itertools

print("==== BTrack Parameters Grid Search ====")

# 1. Define combinations to test
# 3*3*3*3 = 81 iterations. Depending on song length this might run heavily!
# Shrink these down if you only want to test a couple at a time.

momentums = [ 6, 10 , 15]
pulls     = [0.001,0.005,0.01]
latencies = [ 0.040]
decays    = [0.98]

best_score = 0
best_params = {}

for m, p, l, d in itertools.product(momentums, pulls, latencies, decays):
    score = test_parameters(m, p, l, d)
    
    print(f"Tested [Momentum: {m:<3} | Pull: {p:<4} | Latency: {l:<5} | Decay: {d:<4}] -> F1: {score*100:.2f}%")
    
    if score > best_score:
        best_score = score
        best_params = {'momentum': m, 'pull': p, 'latency': l, 'decay': d}
        print(f"\t⭐ NEW BEST!")
        
print("\n===========================")
print(f"✅ FINISHED! Best F1 Score found: {best_score*100:.2f}%")
print(f"Optimal Parameters:\n{best_params}")


==== BTrack Parameters Grid Search ====
Tested [Momentum: 6   | Pull: 0.001 | Latency: 0.04  | Decay: 0.98] -> F1: 84.91%
	⭐ NEW BEST!
Tested [Momentum: 6   | Pull: 0.005 | Latency: 0.04  | Decay: 0.98] -> F1: 84.87%
Tested [Momentum: 6   | Pull: 0.01 | Latency: 0.04  | Decay: 0.98] -> F1: 85.20%
	⭐ NEW BEST!
Tested [Momentum: 10  | Pull: 0.001 | Latency: 0.04  | Decay: 0.98] -> F1: 84.71%
Tested [Momentum: 10  | Pull: 0.005 | Latency: 0.04  | Decay: 0.98] -> F1: 84.80%
Tested [Momentum: 10  | Pull: 0.01 | Latency: 0.04  | Decay: 0.98] -> F1: 84.62%


KeyboardInterrupt: 